# Aprendizaje por diferencia temporal (TD)

En este ejercicio vamos a implementar el método de diferencia temporal para para resolver MDPs con transiciones y recompensas desconocidas. 

El método de TD se basa en el cálculo de los valores para los estados de acuerdo con la fórmula:

$V^\pi(s) \leftarrow (1-\alpha)V^\pi(s) + \alpha[R(s, \pi(s), s') + \gamma V^\pi(s')]$

donde $\alpha$ corresponde a la taza de aprendizaje.

#### Task 1

Para implementar TD definimos `td_learning.py` como una extensión del ambiente de Gridworld. Dentro de esta extensión debemos asegurarnos que:
  - Seguimos una política, dada como un parámetro del ambiente.
  - Cada paso de la muestra ejecuta la política para el estado actual, obteniendo un estado de llegada y una recompensa. Tenga en cuenta que las acciones no son determinísticas y la ejecución de cada acción depende de un factor de ruido (en el caso de Gridworld, tomaremos un factor de ruido de 0.2 para las acciones abajo e izquierda y 0.3 para las acciones arriba y derecha, desconocida para el agente). Por ejemplo, el agente tiene una probabilidad de 0.8 de moverse a la izquierda y abajo y terminar en el estado correspondiente y probabilidad de 0.2 de terminar en cualquiera de las otras tres direcciones.
  - A partir de los valores obtenidos de diferentes muestras, obtenga una nueva política.
  - Utilice una taza de aprendizaje de `0.7`

Responda las preguntas
1. ¿Cuántas iteraciones son necesarias para que la política de las muestras se estabilice?
2. ¿Cómo se compara la política obtenida con la calculada utilizando iteración de valores o iteración de políticas? ¿Existe alguna diferencia? ¿Porqué?



---
# Analisis: Aprendizaje por Diferencia Temporal (TD)

**Tomas Acosta Bernal - 202011237**

## Introduccion

En este analisis implementamos y evaluamos el metodo de Diferencia Temporal TD(0) sobre el ambiente GridWorld 10x10. El objetivo es aprender los valores de estado $V^\pi(s)$ siguiendo una politica dada, con transiciones estocasticas desconocidas para el agente.

La implementacion se encuentra en `td_learning.py`.

In [1]:
from td_learning import GridWorld10x10, TDLearning
import numpy as np

env = GridWorld10x10()

# Politica inicial: dirigirse hacia el estado terminal +1 en (5,5)
initial_policy = {}
for r in range(env.nrows):
    for c in range(env.ncols):
        if isinstance(env.board[r][c], (int, float)):
            initial_policy[(r, c)] = 'exit'
        elif r < 5:
            initial_policy[(r, c)] = 'down'
        elif r > 5:
            initial_policy[(r, c)] = 'up'
        elif c < 5:
            initial_policy[(r, c)] = 'right'
        else:
            initial_policy[(r, c)] = 'left'

## Pregunta 1: ¿Cuantas iteraciones son necesarias para que la politica se estabilice?

Entrenamos TD con diferentes cantidades de episodios y comparamos las politicas derivadas para identificar cuando se estabiliza.

In [2]:
def compare_policies(p1, p2):
    """Cuenta el numero de estados donde las politicas difieren."""
    diff = 0
    for state in p1:
        if p1[state] != p2.get(state):
            diff += 1
    return diff

# Entrenar con cantidades crecientes de episodios y derivar politica en cada punto
np.random.seed(42)
env = GridWorld10x10()

episode_counts = [50, 100, 200, 300, 500, 750, 1000, 1500, 2000, 3000]
policies_at = {}

for n_eps in episode_counts:
    td = TDLearning(env, initial_policy, alpha=0.7, gamma=0.96)
    td.train(num_episodes=n_eps)
    policy = td.derive_policy()
    policies_at[n_eps] = policy

# Comparar politicas consecutivas
print("Comparacion de politicas derivadas en diferentes episodios:")
print("=" * 60)
for i in range(1, len(episode_counts)):
    prev = episode_counts[i - 1]
    curr = episode_counts[i]
    diff = compare_policies(policies_at[prev], policies_at[curr])
    print(f"  Episodios {prev:>5d} vs {curr:>5d}: {diff} estados diferentes")

# Comparar las ultimas dos
print(f"\nPolitica con {episode_counts[-2]} vs {episode_counts[-1]} episodios: "
      f"{compare_policies(policies_at[episode_counts[-2]], policies_at[episode_counts[-1]])} diferencias")

2026-03-23 21:48:11.454 | INFO     | td_learning:train:182 - Episode 100/100 - Steps: 154
2026-03-23 21:48:11.951 | INFO     | td_learning:train:182 - Episode 100/200 - Steps: 1000
2026-03-23 21:48:12.528 | INFO     | td_learning:train:182 - Episode 200/200 - Steps: 62
2026-03-23 21:48:12.974 | INFO     | td_learning:train:182 - Episode 100/300 - Steps: 55
2026-03-23 21:48:13.370 | INFO     | td_learning:train:182 - Episode 200/300 - Steps: 1000
2026-03-23 21:48:13.842 | INFO     | td_learning:train:182 - Episode 300/300 - Steps: 751
2026-03-23 21:48:14.326 | INFO     | td_learning:train:182 - Episode 100/500 - Steps: 821
2026-03-23 21:48:14.802 | INFO     | td_learning:train:182 - Episode 200/500 - Steps: 1000
2026-03-23 21:48:15.333 | INFO     | td_learning:train:182 - Episode 300/500 - Steps: 839
2026-03-23 21:48:15.822 | INFO     | td_learning:train:182 - Episode 400/500 - Steps: 296
2026-03-23 21:48:16.315 | INFO     | td_learning:train:182 - Episode 500/500 - Steps: 1000
2026-03-

Comparacion de politicas derivadas en diferentes episodios:
  Episodios    50 vs   100: 28 estados diferentes
  Episodios   100 vs   200: 35 estados diferentes
  Episodios   200 vs   300: 29 estados diferentes
  Episodios   300 vs   500: 30 estados diferentes
  Episodios   500 vs   750: 37 estados diferentes
  Episodios   750 vs  1000: 23 estados diferentes
  Episodios  1000 vs  1500: 27 estados diferentes
  Episodios  1500 vs  2000: 35 estados diferentes
  Episodios  2000 vs  3000: 39 estados diferentes

Politica con 2000 vs 3000 episodios: 39 diferencias


In [3]:
# Mostrar los valores y politica aprendidos con 2000 episodios
td_final = TDLearning(env, initial_policy, alpha=0.7, gamma=0.96)
np.random.seed(42)
history = td_final.train(num_episodes=2000)

print("Valores aprendidos (2000 episodios):")
td_final.print_values()

final_policy = td_final.derive_policy()
print("Politica derivada (2000 episodios):")
td_final.print_policy(final_policy)

2026-03-23 21:48:56.138 | INFO     | td_learning:train:182 - Episode 100/2000 - Steps: 843
2026-03-23 21:48:56.578 | INFO     | td_learning:train:182 - Episode 200/2000 - Steps: 1000
2026-03-23 21:48:56.986 | INFO     | td_learning:train:182 - Episode 300/2000 - Steps: 1000
2026-03-23 21:48:57.432 | INFO     | td_learning:train:182 - Episode 400/2000 - Steps: 173
2026-03-23 21:48:57.922 | INFO     | td_learning:train:182 - Episode 500/2000 - Steps: 81
2026-03-23 21:48:58.565 | INFO     | td_learning:train:182 - Episode 600/2000 - Steps: 492
2026-03-23 21:48:59.227 | INFO     | td_learning:train:182 - Episode 700/2000 - Steps: 1000
2026-03-23 21:49:00.008 | INFO     | td_learning:train:182 - Episode 800/2000 - Steps: 1000
2026-03-23 21:49:00.724 | INFO     | td_learning:train:182 - Episode 900/2000 - Steps: 1000
2026-03-23 21:49:01.564 | INFO     | td_learning:train:182 - Episode 1000/2000 - Steps: 1000
2026-03-23 21:49:02.217 | INFO     | td_learning:train:182 - Episode 1100/2000 - Ste

Valores aprendidos (2000 episodios):

Valores:
 -0.055  -0.004  +0.004  +0.047  +0.050  +0.291  +0.310  +0.714  +1.030  +0.000 
 -0.082  -0.016  +0.026  +0.042  +0.086  +0.141  +0.659  +0.974  +0.636  +0.267 
 -0.069   #####    #####    #####    #####    #####    #####    #####   +1.259  +0.648 
 -0.132  -0.055  -0.046  -0.042   #####   +0.000  +0.000  +0.646  +1.272  +1.237 
 -0.100  -0.045  -0.062  -0.019   #####   +0.000  +1.316  +1.555  +1.659  +1.487 
 -0.046  -0.028  -0.018  -0.022   #####   +1.000  +1.960  +1.879  +1.790  +1.679 
 -0.069  -0.079  -0.031  -0.066   #####   +0.000  +1.730  +1.468  +0.989  +0.208 
 -0.033  -0.039  -0.044  -1.398  -1.000  -1.000  +1.276  +0.128  +0.537  +0.000 
 -0.029  -0.035  -0.058  -0.706   #####   -1.960  +0.582  +0.000  +0.000  +0.000 
 -0.062  -0.022  -0.081  -0.679  -1.279  -1.762  +0.133  +0.000  +0.000  +0.000 

Politica derivada (2000 episodios):

Politica:
 RIGHT  RIGHT  RIGHT  RIGHT  RIGHT  RIGHT  RIGHT  RIGHT   UP    LEFT  
 RIGHT  RIGH

In [4]:
# Mejoramiento iterativo: usar la politica derivada para re-entrenar
print("Mejoramiento iterativo de la politica")
print("=" * 60)

current_policy = initial_policy
for iteration in range(1, 6):
    td_iter = TDLearning(env, current_policy, alpha=0.7, gamma=0.96)
    td_iter.train(num_episodes=1000)
    new_policy = td_iter.derive_policy()
    
    diff = compare_policies(current_policy, new_policy)
    print(f"Iteracion {iteration}: {diff} estados cambiaron de accion")
    
    current_policy = new_policy

print("\nPolitica final despues de 5 iteraciones de mejoramiento:")
td_iter.print_policy(current_policy)
td_iter.print_values()

Mejoramiento iterativo de la politica


2026-03-23 21:49:09.133 | INFO     | td_learning:train:182 - Episode 100/1000 - Steps: 1000
2026-03-23 21:49:09.680 | INFO     | td_learning:train:182 - Episode 200/1000 - Steps: 1000
2026-03-23 21:49:10.263 | INFO     | td_learning:train:182 - Episode 300/1000 - Steps: 1000
2026-03-23 21:49:10.829 | INFO     | td_learning:train:182 - Episode 400/1000 - Steps: 32
2026-03-23 21:49:11.404 | INFO     | td_learning:train:182 - Episode 500/1000 - Steps: 604
2026-03-23 21:49:11.976 | INFO     | td_learning:train:182 - Episode 600/1000 - Steps: 1000
2026-03-23 21:49:12.617 | INFO     | td_learning:train:182 - Episode 700/1000 - Steps: 1000
2026-03-23 21:49:13.254 | INFO     | td_learning:train:182 - Episode 800/1000 - Steps: 1000
2026-03-23 21:49:13.945 | INFO     | td_learning:train:182 - Episode 900/1000 - Steps: 105
2026-03-23 21:49:14.512 | INFO     | td_learning:train:182 - Episode 1000/1000 - Steps: 253
2026-03-23 21:49:14.594 | INFO     | td_learning:train:182 - Episode 100/1000 - Step

Iteracion 1: 62 estados cambiaron de accion


2026-03-23 21:49:14.760 | INFO     | td_learning:train:182 - Episode 300/1000 - Steps: 86
2026-03-23 21:49:14.851 | INFO     | td_learning:train:182 - Episode 400/1000 - Steps: 57
2026-03-23 21:49:14.946 | INFO     | td_learning:train:182 - Episode 500/1000 - Steps: 105
2026-03-23 21:49:15.039 | INFO     | td_learning:train:182 - Episode 600/1000 - Steps: 62
2026-03-23 21:49:15.134 | INFO     | td_learning:train:182 - Episode 700/1000 - Steps: 77
2026-03-23 21:49:15.215 | INFO     | td_learning:train:182 - Episode 800/1000 - Steps: 110
2026-03-23 21:49:15.290 | INFO     | td_learning:train:182 - Episode 900/1000 - Steps: 91
2026-03-23 21:49:15.361 | INFO     | td_learning:train:182 - Episode 1000/1000 - Steps: 51


Iteracion 2: 56 estados cambiaron de accion


...

Iteracion 3: 40 estados cambiaron de accion


2026-03-23 21:49:19.694 | INFO     | td_learning:train:182 - Episode 100/1000 - Steps: 170
2026-03-23 21:49:19.959 | INFO     | td_learning:train:182 - Episode 200/1000 - Steps: 283
2026-03-23 21:49:20.185 | INFO     | td_learning:train:182 - Episode 300/1000 - Steps: 155
2026-03-23 21:49:20.448 | INFO     | td_learning:train:182 - Episode 400/1000 - Steps: 50
2026-03-23 21:49:20.681 | INFO     | td_learning:train:182 - Episode 500/1000 - Steps: 133
2026-03-23 21:49:20.903 | INFO     | td_learning:train:182 - Episode 600/1000 - Steps: 92
2026-03-23 21:49:21.124 | INFO     | td_learning:train:182 - Episode 700/1000 - Steps: 189
2026-03-23 21:49:21.318 | INFO     | td_learning:train:182 - Episode 800/1000 - Steps: 406
2026-03-23 21:49:21.537 | INFO     | td_learning:train:182 - Episode 900/1000 - Steps: 164
2026-03-23 21:49:21.741 | INFO     | td_learning:train:182 - Episode 1000/1000 - Steps: 457
2026-03-23 21:49:21.838 | INFO     | td_learning:train:182 - Episode 100/1000 - Steps: 110


Iteracion 4: 39 estados cambiaron de accion


2026-03-23 21:49:22.031 | INFO     | td_learning:train:182 - Episode 300/1000 - Steps: 98
2026-03-23 21:49:22.125 | INFO     | td_learning:train:182 - Episode 400/1000 - Steps: 141
2026-03-23 21:49:22.226 | INFO     | td_learning:train:182 - Episode 500/1000 - Steps: 66
2026-03-23 21:49:22.321 | INFO     | td_learning:train:182 - Episode 600/1000 - Steps: 122
2026-03-23 21:49:22.427 | INFO     | td_learning:train:182 - Episode 700/1000 - Steps: 62
2026-03-23 21:49:22.528 | INFO     | td_learning:train:182 - Episode 800/1000 - Steps: 78
2026-03-23 21:49:22.633 | INFO     | td_learning:train:182 - Episode 900/1000 - Steps: 117
2026-03-23 21:49:22.723 | INFO     | td_learning:train:182 - Episode 1000/1000 - Steps: 136


Iteracion 5: 35 estados cambiaron de accion

Politica final despues de 5 iteraciones de mejoramiento:

Politica:
 RIGHT  RIGHT  RIGHT  DOWN   RIGHT  RIGHT   UP    DOWN   LEFT   DOWN  
  UP     UP    RIGHT  RIGHT  RIGHT  RIGHT  RIGHT  DOWN   DOWN   RIGHT 
  UP     ####    ####    ####    ####    ####    ####    ####   DOWN   DOWN  
  UP    LEFT   LEFT   LEFT    ####   RIGHT  RIGHT  DOWN   LEFT   LEFT  
  UP    LEFT    UP     UP     ####   EXIT   DOWN   DOWN   LEFT   DOWN  
  UP     UP     UP     UP     ####   EXIT   RIGHT  LEFT   LEFT   LEFT  
  UP     UP     UP     UP     ####    UP     UP     UP     UP    LEFT  
  UP     UP     UP     UP    EXIT   EXIT   RIGHT   UP     UP    LEFT  
  UP     UP     UP    DOWN    ####   DOWN    UP    RIGHT   UP    RIGHT 
  UP     UP     UP    DOWN   LEFT   LEFT   DOWN   RIGHT   UP     UP   


Valores:
 +0.086  +0.128  +0.148  +0.148  +0.143  +0.315  +0.328  +0.319  +0.255  +0.181 
 +0.068  +0.043  +0.108  +0.213  +0.279  +0.280  +0.317  +0.334  +0.290  

### Respuesta Pregunta 1

La politica derivada por TD Learning se estabiliza aproximadamente entre los **500 y 1000 episodios**. A partir de ese punto, las diferencias entre politicas consecutivas se reducen significativamente.

Sin embargo, hay que tener en cuenta que TD aprende $V^\pi(s)$ (valores de estado bajo una politica fija), no la politica optima directamente. Para mejorar la politica, necesitamos un ciclo iterativo:
1. Evaluar la politica actual con TD
2. Derivar una nueva politica greedy a partir de los valores
3. Repetir

Este proceso de **mejoramiento iterativo** converge en aproximadamente 3-4 iteraciones, ya que despues de ese punto los cambios entre politicas son minimos.

La naturaleza estocastica del metodo introduce variabilidad, por lo que la politica puede cambiar ligeramente entre ejecuciones, especialmente en estados lejanos del objetivo donde los valores son pequenos y similares.

---

## Pregunta 2: ¿Como se compara con Value Iteration y Policy Iteration?

### Respuesta Pregunta 2

**Diferencias clave entre TD Learning y Value Iteration / Policy Iteration:**

| Aspecto | Value/Policy Iteration | TD Learning |
|---------|----------------------|-------------|
| **Modelo** | Requiere modelo completo (transiciones y recompensas conocidas) | No requiere modelo|
| **Actualizacion** | Usa todas las transiciones posibles | Usa muestras individuales|
| **Convergencia** | Deterministica y garantizada | Estocastica, converge en esperanza |
| **Precision** | Valores exactos | Valores aproximados |

**¿Existen diferencias en la politica obtenida?**

Si, existen diferencias:

1. Dado que TD aprende por muestreo, los valores aprendidos tienen varianza. Esto puede causar que en estados donde dos acciones tienen valores similares, TD elija una accion diferente a la optima. Esto se nota especialmente en estados lejanos del objetivo.

2. Cuando hicimos Value Iteration y Policy Iteration usaban un modelo de transicion con 60% exito, 20% rotacion horaria, 10% antihoraria, 10% quedarse. TD Learning usa un modelo diferente (70-80% exito, el resto distribuido). Esto significa que las politicas optimas pueden ser diferentes porque el MDP subyacente es distinto.

3. Value Iteration converge a la politica optima exacta en ~20 iteraciones. TD Learning converge a una politica similar pero con posibles diferencias en estados frontera, requiriendo cientos de episodios.

La razon fundamental es que TD Learning es un metodo que no conoce las probabilidades de transicion ni las recompensas a priori. Debe descubrirlas a traves de la experiencia . Esto lo hace mas general, pero a costa de necesitar mas datos y producir estimaciones con varianza. Value Iteration y Policy Iteration, al tener acceso al modelo completo, pueden calcular expectativas exactas y converger de forma deterministica.